# 01 — Data exploration (data v1)

Reads the cleaned data built by `scripts/prepare_data.py --version v1` (Parquet, manifest,
cleaning log). It creates no data; it only draws the charts saved to `reports/figures/` and
records what they show. Column names are the verified ones from `COLUMN_MAP` (E00): A1 and B
hold the paper's features (fiat/biat as min, max, mean, std), A2 holds `total_fiat` /
`total_biat` instead of the two std columns.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from blindspy.data import CORE_TASKS, LABELS, ROOT, TIMEOUTS, processed_dir

V1 = processed_dir("v1")
FIG = ROOT / "reports" / "figures"
manifest = json.loads((V1 / "manifest.json").read_text())
data = {(t, s): pd.read_parquet(V1 / f"{t}_{s}s.parquet") for t in CORE_TASKS for s in TIMEOUTS}
log = pd.read_csv(V1 / "cleaning_log.csv")

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#1f1f1e", "#6b6a64"
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": MUTED,
        "axes.labelcolor": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "axes.grid": True,
        "grid.color": "#e6e5e0",
        "grid.linewidth": 0.6,
        "axes.axisbelow": True,
        "font.size": 9,
    }
)

## 1. Dataset summary per task and timeout

In [ ]:
summary = pd.DataFrame({k: v["summary"] for k, v in manifest["tasks"].items()}).T
summary

**Findings.** All 16 core files (4 tasks x 4 timeouts) have 23 features, 0 missing values and 0 infinite values after cleaning. Final sizes run from 4,648 flows (A2-VPN, 120 s) to 18,074 (B, 15 s). A1 is nearly balanced (largest-to-smallest class 1.03:1 to 1.26:1); the 7- and 14-class tasks are not, up to 32.47:1 in A2-NonVPN at 30 s (BROWSING 2,500 vs MAIL 77). Macro F1, not accuracy, is therefore the main metric.

## 2. Class counts per task (data v1, after cleaning)

In [ ]:
fig, axes = plt.subplots(len(CORE_TASKS), len(TIMEOUTS), figsize=(13, 11), sharey="row")
for i, task in enumerate(CORE_TASKS):
    for j, t in enumerate(TIMEOUTS):
        ax = axes[i, j]
        counts = data[(task, t)]["label"].value_counts().reindex(LABELS[task], fill_value=0)
        ax.barh(counts.index, counts.values, color=BLUE, height=0.7)
        for y, v in enumerate(counts.values):
            ax.text(v, y, f" {v:,}", va="center", fontsize=7, color=INK)
        ax.set_title(f"{task}, {t} s ({counts.sum():,} flows)", fontsize=9, color=INK)
        ax.invert_yaxis()
        ax.grid(axis="y", visible=False)
        ax.set_xlim(0, counts.max() * 1.3)
fig.suptitle("Flows per class after cleaning (data v1)", x=0.01, ha="left", color=INK)
fig.tight_layout()
fig.savefig(FIG / "class_counts_v1.png")

**Findings.** BROWSING and VPN-BROWSING hold exactly 2,500 flows in every raw A2 file, which points to the dataset authors capping that class; after de-duplication VPN-BROWSING drops to 2,463 (60 s) and 2,244 (120 s). The smallest classes are MAIL at 30 s (77 flows) and VPN-STREAMING at 120 s (145). B's 14 class counts equal the A2-VPN and A2-NonVPN counts at the same timeout, because B and A2 hold the same flows (E00).

## 3. Timeout comparability

Same 23 feature names and order at every timeout within a task, same units (times in
microseconds, rates per second); what changes is the number of flows and the class mix.

In [ ]:
tt = manifest["timeouts"]
same_features = {
    task: len({tuple(tt[task][f"{t}s"]["features_in_order"]) for t in TIMEOUTS}) == 1
    for task in CORE_TASKS
}
print("feature names and order identical across timeouts:", same_features)
rows = pd.DataFrame(
    {task: {f"{t} s": tt[task][f"{t}s"]["final_rows"] for t in TIMEOUTS} for task in CORE_TASKS}
)
rows

In [ ]:
fig, axes = plt.subplots(
    1, len(CORE_TASKS), figsize=(15, 6.5), gridspec_kw={"width_ratios": [0.6, 1, 1, 1.4]}
)
for ax, task in zip(axes, CORE_TASKS, strict=True):
    mix = pd.DataFrame({f"{t} s": tt[task][f"{t}s"]["class_mix_pct"] for t in TIMEOUTS})
    mix = mix.reindex(LABELS[task]).fillna(0)
    sns.heatmap(
        mix,
        ax=ax,
        cmap=sns.light_palette(BLUE, as_cmap=True),
        vmin=0,
        vmax=50,
        annot=True,
        fmt=".1f",
        annot_kws={"fontsize": 7},
        cbar=False,
        linewidths=1,
        linecolor="white",
    )
    ax.set_title(task, color=INK)
    ax.tick_params(length=0)
    ax.tick_params(axis="y", rotation=0)
    ax.grid(False)
fig.suptitle("Class mix (% of flows) at each flow timeout, data v1", x=0.01, ha="left", color=INK)
fig.tight_layout()
fig.savefig(FIG / "timeout_class_mix.png")
spread = pd.DataFrame(
    {
        task: pd.DataFrame({t: tt[task][f"{t}s"]["class_mix_pct"] for t in TIMEOUTS}).pipe(
            lambda d: d.max(axis=1) - d.min(axis=1)
        )
        for task in CORE_TASKS
    }
)
print("largest change in a class's share across timeouts (percentage points):")
spread.max().round(1)

**Findings.** Within each task the 23 feature names, their order and their units (times in microseconds, rates per second) are the same at all four timeouts. Two things change. Flow counts fall as the timeout grows (A1: 18,038 at 15 s to 9,524 at 120 s). The class mix also moves a lot: a class's share changes by up to 24.6 percentage points in A2-NonVPN (VOIP 32.5% at 15 s vs 7.9% at 120 s; BROWSING 28.8% vs 50.6%), 21.6 in A2-VPN, 12.2 in B and 7.6 in A1. Any score difference between timeouts is therefore confounded by class mix; the report must say so instead of crediting the timeout. The 60 s files also differ in what VOIP and FT flows look like (section 6), which is a second reason not to read timeout trends causally.

## 4. Data funnel (cleaning log)

Rows kept and removed per file. Only two steps remove rows: the parser skipping comma-only
lines (1 line in each 60 s file of A1 and B) and dropping exact duplicates. Every other step
is logged with the rows it applies to and changes nothing.

In [ ]:
steps = log.sort_values(["task", "timeout", "step"])
funnel = (
    steps.groupby(["task", "timeout"])
    .agg(
        raw=("rows_before", "first"), final=("rows_after", "last"), removed=("rows_removed", "sum")
    )
    .reindex(pd.MultiIndex.from_product([CORE_TASKS, TIMEOUTS]))
)
funnel["removed_pct"] = (funnel["removed"] / funnel["raw"] * 100).round(2)
labels = [f"{t} {s} s" for t, s in funnel.index]
fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(funnel))
ax.barh(y, funnel["final"], color=BLUE, height=0.7, label="kept (final rows)")
ax.barh(
    y,
    funnel["removed"],
    left=funnel["final"] + 60,
    color=ORANGE,
    height=0.7,
    label="removed (duplicates + comma-only lines)",
)
cols = zip(funnel["final"], funnel["removed"], funnel["removed_pct"], strict=True)
for yi, (f, r, p) in enumerate(cols):
    ax.text(f + r + 200, yi, f"{r:,} removed ({p}%)", va="center", fontsize=7, color=INK)
ax.set_yticks(y, labels)
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.set_xlabel("flows")
ax.set_xlim(0, funnel["raw"].max() * 1.25)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
ax.set_title("Data funnel, data v1: raw rows -> final rows per file", loc="left", color=INK, pad=28)
fig.savefig(FIG / "data_funnel_v1.png")
funnel

In [ ]:
per_class = (
    pd.DataFrame({k: v["exact_duplicates_removed_per_class"] for k, v in manifest["tasks"].items()})
    .fillna(0)
    .astype(int)
)
per_class

In [ ]:
fig, axes = plt.subplots(len(CORE_TASKS), len(TIMEOUTS), figsize=(14, 13), sharey="row")
for i, task in enumerate(CORE_TASKS):
    for j, t in enumerate(TIMEOUTS):
        ax = axes[i, j]
        steps = log[(log.task == task) & (log.timeout == t)].set_index("step_name")
        raw = pd.Series(json.loads(steps.loc["parse", "rows_after_per_class"]))
        kept = pd.Series(json.loads(steps.loc["no_resampling", "rows_after_per_class"]))
        raw, kept = (
            raw.reindex(LABELS[task], fill_value=0),
            kept.reindex(LABELS[task], fill_value=0),
        )
        y = np.arange(len(raw))
        ax.barh(y, kept, color=BLUE, height=0.7, label="kept")
        ax.barh(
            y,
            raw - kept,
            left=kept + raw.max() * 0.01,
            color=ORANGE,
            height=0.7,
            label="removed (exact duplicates)",
        )
        for yi, (k, r) in enumerate(zip(kept, raw, strict=True)):
            if r > k:
                ax.text(
                    r * 1.02 + raw.max() * 0.01,
                    yi,
                    f"-{r - k:,}",
                    va="center",
                    fontsize=6.5,
                    color=INK,
                )
        ax.set_yticks(y, raw.index, fontsize=7)
        ax.invert_yaxis()
        ax.grid(axis="y", visible=False)
        ax.set_xlim(0, raw.max() * 1.3)
        ax.set_title(f"{task}, {t} s", fontsize=9, color=INK)
axes[0, 0].legend(loc="lower left", bbox_to_anchor=(0, 1.15), ncol=2, frameon=False)
fig.suptitle(
    "Data funnel by class, data v1: rows kept and removed per class", x=0.01, ha="left", color=INK
)
fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig(FIG / "data_funnel_by_class_v1.png")

**Findings.** Only two steps remove rows: the parser skips one comma-only line in each 60 s file of A1 and B, and exact de-duplication removes 3.02% (A2-NonVPN, 15 s) to 17.46% (A2-VPN, 120 s). Removals are highest at 120 s and are not spread evenly across classes. VPN-MAIL loses 483 of 848 flows at 30 s and 420 of 722 at 120 s; VPN-P2P loses 262 at 15 s; FT loses 160-171 at every timeout. In A1 at 120 s, 1,042 VPN rows go against 216 Non-VPN. Per-class removals are logged in cleaning_log.csv and summarised in the manifest. The by-class chart (data_funnel_by_class_v1.png) shows the same per class and per timeout: VPN-P2P loses 215-262 flows at every timeout, VPN-STREAMING and BROWSING lose none.

## 5. Odd values per feature

Counts in the cleaned 15 s files (the other timeouts follow the same pattern; all counts are
in `data/manifest.json`). -1 means "not measurable"; every value is kept as-is (E00).

In [ ]:
def odd_count(kind):
    return pd.DataFrame(
        {
            task: {f: v[kind] for f, v in manifest["tasks"][f"{task}_15s"]["odd_values"].items()}
            for task in CORE_TASKS
        }
    )


odd, zero = odd_count("exactly_-1"), odd_count("zero")
pd.concat({"count of -1": odd, "count of 0": zero}, axis=1).astype("Int64")

**Findings.** The -1 codes are frequent and structured. At 15 s, 2,022 of 18,038 A1 flows (11.2%) have no forward gap (min_fiat = max_fiat = -1, mean_fiat = 0), and 10,635 (59.0%) have no active/idle period. Zeros in std_fiat/std_biat include those -1-coded flows plus flows with exactly two packets in that direction. Every value is kept as-is (E00 rule), so tree models can split on the codes and scaled models see them as a value just below 0.

## 6. Feature distributions by class (15 s)

Box plots on a symmetric-log axis, because most features span 5-9 orders of magnitude and
include -1 codes. One figure per task and timeout (16 figures, saved as
`feature_boxes_<task>_<timeout>s.png`); the median table below is for 15 s.

In [ ]:
T = 15


def boxes(task, t=T):
    df = data[(task, t)]
    feats = list(df.columns[:-1])
    order = [c for c in LABELS[task] if c in set(df["label"])]
    fig, axes = plt.subplots(6, 4, figsize=(14, 19))
    for ax, f in zip(axes.flat, feats, strict=False):  # 24 panels, 23 features
        sns.boxplot(
            data=df,
            x=f,
            y="label",
            order=order,
            ax=ax,
            color="#cde2fb",
            linecolor=BLUE,
            linewidth=0.8,
            fliersize=1,
            flierprops={"alpha": 0.3},
        )
        ax.set_xscale("symlog", linthresh=1)
        low = df[f].min()
        ticks = [0, 1e2, 1e4, 1e6, 1e8] if low >= 0 else [-1, 0, 1e2, 1e4, 1e6, 1e8]
        if low < -1:
            ticks = [-1e3, *ticks]
        ax.set_xlim(left=low * 1.5 if low < 0 else -0.5)
        ax.set_xticks(
            ticks,
            [
                f"{t:g}"
                if abs(t) < 100
                else f"1e{int(np.log10(abs(t)))}"
                if t > 0
                else f"-1e{int(np.log10(abs(t)))}"
                for t in ticks
            ],
        )
        ax.set_title(f, fontsize=9, color=INK)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.tick_params(labelsize=7)
    for ax in axes.flat[len(feats) :]:
        ax.set_visible(False)
    fig.suptitle(
        f"{task}, {t} s: feature distributions by class (symlog axis; times in us)",
        x=0.01,
        ha="left",
        color=INK,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    fig.savefig(FIG / f"feature_boxes_{task}_{t}s.png")


for task in CORE_TASKS:
    for t in TIMEOUTS:
        boxes(task, t)

In [ ]:
# Median of each feature per class (15 s), to read the box plots by number.
medians = {
    task: data[(task, T)].groupby("label").median().T for task in ["A1", "A2-VPN", "A2-NonVPN"]
}
pd.concat(medians, axis=1).round(0)

**Findings.** Classes differ by orders of magnitude on rate and gap features, which is what the paper's approach relies on. At 15 s: VPN-STREAMING median 202 packets/s and 245 kB/s vs 2.2 packets/s for VPN-VOIP. Non-VPN FT reaches 1,237 packets/s with a 1.0 ms median forward gap, while VPN-FT has 7.1 packets/s and 193 ms. Non-VPN VOIP has a 20 ms median forward gap (a steady audio rhythm), but VPN-VOIP sits at 960 ms with almost no spread (min_fiat median 959 ms). In A1, VPN flows have a 10x larger median mean_fiat (209 ms vs 20 ms) and a 5x lower packet rate (8.9 vs 44.9 packets/s). The share of -1 codes also differs by class: no active/idle period for 99.4% of VOIP vs 29.9% of BROWSING flows, and no forward gap for 79.2% of P2P flows. Distributions overlap heavily inside VPN classes (BROWSING, CHAT, MAIL), so those pairs are the likely confusions. Drawn for all four timeouts (16 figures). The 15 s patterns hold at 30 s and 120 s but not at 60 s, where two classes look different (reports/data_checks_v1.json, voip_ft_profile_per_timeout). Non-VPN VOIP drops from about 100 packets/s and a 20 ms median gap at 15/30/120 s to 2.1 packets/s and 962 ms at 60 s. VPN-VOIP moves the other way (16.5 packets/s, 18 ms at 60 s; 2.1-2.2 packets/s and about 960 ms elsewhere). Non-VPN FT falls from 1,237-3,739 to 56.5 packets/s. It is not a label swap: of 7,041 flows found identically in more than one timeout file, 210 change label between timeouts, mostly 2-packet flows, and no VoIP pair is among them. The 60 s files are also the only ones exported with the paper-style header, so they may come from a different export run. The cause can't be settled from the ARFF files, so 60 s results must be read with this in mind (report: Limitations).

## 7. Correlation between features (15 s)

Spearman rank correlation (robust to the heavy tails). A1/B and A2 have different feature sets,
so each family gets its own heatmaps; all four tasks are drawn at 15 s.

In [ ]:
def corr_plot(task, t=T):
    c = data[(task, t)].drop(columns="label").corr(method="spearman")
    fig, ax = plt.subplots(figsize=(10, 8.5))
    cmap = sns.diverging_palette(250, 25, s=80, l=55, center="light", as_cmap=True)
    sns.heatmap(
        c,
        ax=ax,
        cmap=cmap,
        vmin=-1,
        vmax=1,
        center=0,
        square=True,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"shrink": 0.7, "label": "Spearman rho"},
    )
    ax.set_title(f"{task}, {t} s: Spearman correlation of the 23 features", loc="left", color=INK)
    ax.tick_params(length=0, labelsize=8)
    ax.grid(False)
    fig.savefig(FIG / f"correlation_{task}_{t}s.png")
    return c


corr = {task: corr_plot(task) for task in CORE_TASKS}


def top_pairs(c, n=12):
    upper = c.where(np.triu(np.ones(c.shape, bool), 1)).stack().dropna()
    return upper.reindex(upper.abs().sort_values(ascending=False).index).head(n).round(3)


pd.concat({task: top_pairs(c).reset_index() for task, c in corr.items()}, axis=1)

In [ ]:
for task, c in corr.items():
    upper = c.where(np.triu(np.ones(c.shape, bool), 1)).stack().dropna().abs()
    print(task, "pairs with |rho| >= 0.9:", int((upper >= 0.9).sum()), "of", len(upper))

**Findings.** Many features are near-duplicates. 19 of 253 feature pairs in A1 and B, 27 in A2-VPN and 20 in A2-NonVPN have Spearman |rho| >= 0.9, almost all inside the active/idle block (e.g. std_active vs std_idle 0.996, mean_idle vs max_idle 0.996), plus flowPktsPerSecond vs mean_flowiat (-0.990, which follows from rate = 1 / mean gap). Feature importances and SHAP values will therefore be shared out among correlated features, so they describe model behaviour, not causes (as the plan's Metrics section notes). E09's group ablation is the better test of what each group adds.